In [8]:
pip install pandas pyarrow pyyaml pandera

^C
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import json
import os
from pathlib import Path

import pandas as pd
import yaml
import pandera as pa
from pandera import Check, Column, DataFrameSchema

BASE = Path(os.getcwd())
CFG = BASE / "config" / "pipeline.yaml"

print("Current working folder:", BASE)
print("Config file path:", CFG)

In [ ]:
def load_cfg():
    with open(CFG, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(c).strip().lower().replace(" ", "_") for c in df.columns]
    return df


def clean_strings(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for c in df.columns:
        if df[c].dtype == "object":
            df[c] = df[c].astype(str).str.strip()
            df.loc[df[c].isin(["", "nan", "None", "null"]), c] = pd.NA
    return df


def coerce_datetime(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for c in df.columns:
        lc = c.lower()
        if any(k in lc for k in ["time", "date", "timestamp"]):
            try:
                converted = pd.to_datetime(df[c], errors="coerce")
                if converted.notna().sum() > 0:
                    df[c] = converted
            except Exception:
                pass
    return df


def fill_nulls(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for c in df.columns:
        if pd.api.types.is_numeric_dtype(df[c]):
            df[c] = df[c].fillna(0)
        elif pd.api.types.is_datetime64_any_dtype(df[c]):
            continue
        else:
            df[c] = df[c].fillna("unknown")
    return df

In [ ]:
def build_schema(df: pd.DataFrame) -> DataFrameSchema:
    schema_columns = {}

    for col in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            schema_columns[col] = Column(
                pa.DateTime,
                nullable=True,
                required=True
            )

        elif pd.api.types.is_numeric_dtype(df[col]):
            schema_columns[col] = Column(
                float if "float" in str(df[col].dtype) else int,
                nullable=False,
                required=True
            )

        else:
            schema_columns[col] = Column(
                str,
                nullable=False,
                required=True,
                checks=Check.str_length(min_value=1)
            )

    schema = DataFrameSchema(
        schema_columns,
        strict=False,
        coerce=True
    )

    return schema

In [ ]:
cfg = load_cfg()

raw = BASE / "data" / "raw" / "event_logs.csv"
out_csv = BASE / "data" / "processed" / "cleaned_event_logs.csv"
out_parquet = BASE / "data" / "processed" / "cleaned_event_logs.parquet"
out_json = BASE / "data" / "processed" / "pipeline_summary.json"

print("Raw file:", raw)
print("Output CSV:", out_csv)
print("Output Parquet:", out_parquet)
print("Output JSON:", out_json)

In [ ]:
df = pd.read_csv(raw)

rows_in = len(df)
cols_in = len(df.columns)
nulls_before = int(df.isna().sum().sum())

df = normalize_columns(df)
df = clean_strings(df)
df = coerce_datetime(df)

dup_count = int(df.duplicated().sum())
df = df.drop_duplicates()

df = fill_nulls(df)

schema = build_schema(df)
validated_df = schema.validate(df)

rows_out = len(validated_df)
cols_out = len(validated_df.columns)
nulls_after = int(validated_df.isna().sum().sum())

out_csv.parent.mkdir(parents=True, exist_ok=True)

validated_df.to_csv(out_csv, index=False)
validated_df.to_parquet(out_parquet, index=False)

summary = {
    "project": cfg["project"]["name"],
    "rows_in": rows_in,
    "rows_out": rows_out,
    "columns_in": cols_in,
    "columns_out": cols_out,
    "nulls_before": nulls_before,
    "nulls_after": nulls_after,
    "duplicates_removed": dup_count,
    "sample_columns": list(validated_df.columns[:12]),
    "schema_validation": "passed"
}

with open(out_json, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=4, default=str)

print("Pipeline completed successfully.")
print("Schema validation passed.")
print(f"Cleaned CSV saved to: {out_csv}")
print(f"Cleaned Parquet saved to: {out_parquet}")
print(f"Summary JSON saved to: {out_json}")

In [ ]:
validated_df.head()

In [ ]:
summary